# 00. Dataset Metadata — GSE287331

This notebook extracts and standardizes the phenotypic metadata for dataset GSE287331.  
It retrieves sample-level information from GEO, harmonizes tissue annotations into a 3-class labeling scheme (normal, adjacent/benign, tumor), and stores the final table as a compressed Parquet file for reproducible downstream integration.

**Source: GEO accession GSE287331, platform Illumina Infinium MethylationEPIC v1.0 BeadChip**


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-metadata-GSE28733                       ║
# ║ Description: : Extracts and standardizes phenotypic metadata,    ║
# ║                ensuring consistent sample labeling and           ║
# ║                reproducible downstream integration               ║
# ║ Dataset(s):   GSE287331                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 10-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


## Libraries

In [ ]:
!pip install -q GEOparse polars pyarrow lz4


In [ ]:
import os
import re
from pathlib import Path
import GEOparse
import polars as pl


## 1. Builf pheno_GSE287331

In [ ]:
# BUILD PHENO.parquet for GSE287331 using GEOparse + Polars
# Columns:
#   - id_tissue        (GSM accession)
#   - sample_name
#   - tissue_type_raw
#   - label            (0=HDB, 1=AN, 2=TU, 3=OQ, 4=CUB)
#   - idat_basename

# CONFIG
GSE_ID        = "GSE287331"  
DESTDIR       = "/kaggle/working/geoparse_cache"
OUT_PHENO_PAR = "/kaggle/working/pheno_GSE287331.parquet"

Path(DESTDIR).mkdir(parents=True, exist_ok=True)

# Tissue Mapping (5 classes)
def map_tissue_to_label(tissue: str | None) -> int | None:
    if tissue is None:
        return None
    t = tissue.strip().upper()

    if t == "HDB":                # Healthy donor breast (normal)
        return 0
    if t == "AN":                 # Tumor-adjacent normal
        return 1
    if t in {"TU", "TUMOR"}:      # Tumor
        return 2
    if t == "OQ":                 # Opposite quadrant (ipsilateral)
        return 3
    if t == "CUB":                # Contralateral unaffected breast
        return 4

    return None  # unexpected tissue label

# 1) Load GEO series
gse = GEOparse.get_GEO(
    geo=GSE_ID,
    destdir=DESTDIR,
    annotate_gpl=False,
    how="full",
    silent=True,
)

# 2) Extract per-sample metadata
rows = []

for gsm_name, gsm in gse.gsms.items():
    meta = gsm.metadata

    # GSM ID
    geo_accession = gsm.get_accession()

    # Title → sample_name
    sample_name = meta.get("title", [geo_accession])[0]

    # Tissue extraction from "characteristics_ch1"
    tissue_raw = None
    for val in meta.get("characteristics_ch1", []):
        if val and "tissue:" in val.lower():
            tissue_raw = val.split(":", 1)[1].strip()
            break

    # idat basename (stored in "description")
    desc_list = meta.get("description", [])
    idat_basename = desc_list[0].strip() if desc_list else None

    # Map to label 0–4
    label = map_tissue_to_label(tissue_raw)

    rows.append(
        {
            "id_tissue": geo_accession,
            "sample_name": sample_name,
            "tissue_type_raw": tissue_raw,
            "label": label,
            "idat_basename": idat_basename,
        }
    )

# 3) Build polars DF
pheno = pl.DataFrame(rows)

# Cast label to Int8
pheno = pheno.with_columns(
    pl.col("label").cast(pl.Int8)
)

print("\nHEAD:")
print(pheno.head())

print("\nTISSUE COUNTS:")
print(pheno["tissue_type_raw"].value_counts())

print("\nLABEL COUNTS:")
print(pheno["label"].value_counts())

pheno.write_parquet(
    OUT_PHENO_PAR,
    compression="lz4",
    statistics=True,
)

print(f"\n✅ Saved phenotype table to: {OUT_PHENO_PAR}")
print(f"Shape: {pheno.shape}")
